# Physics-Conditioned ANC Net 训练 Notebook

本 Notebook 提供端到端 PyTorch 训练流程，包含：
1. HDF5 数据加载（`x_gcc`, `x_s_pca`, `y_w_pca`, `V_w`）
2. 双分支融合模型 `PhysicalConditionedANCNet`
3. 物理感知混合损失（系数空间 + 重构空间）
4. 训练/验证循环与每个 Epoch 动态刷新 Loss 曲线
5. 保存最优模型 `best_model.pth`

In [ ]:
import os
import random
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ===============================
# 形状适配工具（兼容不同版本数据）
# ===============================
def resize_1d_sequence(x: np.ndarray, target_len: int) -> np.ndarray:
    """将 [C, L] 的序列特征重采样到目标长度。"""
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError(f'Expect [C, L], got shape={x.shape}')

    c, old_len = x.shape
    if old_len == target_len:
        return x

    old_grid = np.linspace(0.0, 1.0, old_len, dtype=np.float32)
    new_grid = np.linspace(0.0, 1.0, target_len, dtype=np.float32)

    out = np.zeros((c, target_len), dtype=np.float32)
    for i in range(c):
        out[i] = np.interp(new_grid, old_grid, x[i]).astype(np.float32)
    return out

def fit_feature_dim(x: np.ndarray, target_dim: int) -> np.ndarray:
    """将 1D 特征裁剪或零填充到 target_dim。"""
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if x.shape[0] == target_dim:
        return x

    out = np.zeros((target_dim,), dtype=np.float32)
    keep = min(target_dim, x.shape[0])
    out[:keep] = x[:keep]
    return out

def fit_matrix_shape(x: np.ndarray, target_rows: int, target_cols: int) -> np.ndarray:
    """将 2D 矩阵裁剪/零填充到 [target_rows, target_cols]。"""
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError(f'Expect 2D matrix, got shape={x.shape}')

    out = np.zeros((target_rows, target_cols), dtype=np.float32)
    r = min(target_rows, x.shape[0])
    c = min(target_cols, x.shape[1])
    out[:r, :c] = x[:r, :c]
    return out

In [ ]:
# =========================================
# Dataset & DataLoader
# x_gcc: /processed/gcc_phat -> [B, 3, 256]
# x_s_pca: /processed/S_pca_coeffs -> [B, 16]
# y_w_pca: /processed/W_pca_coeffs -> [B, 32]
# =========================================
class ANCHDF5Dataset(Dataset):
    def __init__(
        self,
        h5_path: Path,
        gcc_len: int = 256,
        s_pca_dim: int = 16,
        w_pca_dim: int = 32,
    ):
        self.h5_path = Path(h5_path)
        self.gcc_len = int(gcc_len)
        self.s_pca_dim = int(s_pca_dim)
        self.w_pca_dim = int(w_pca_dim)

        if not self.h5_path.exists():
            raise FileNotFoundError(f'HDF5 file not found: {self.h5_path}')

        with h5py.File(str(self.h5_path), 'r') as f:
            if '/processed/gcc_phat' not in f:
                raise KeyError('Missing /processed/gcc_phat')
            if '/processed/S_pca_coeffs' not in f:
                raise KeyError('Missing /processed/S_pca_coeffs')
            if '/processed/W_pca_coeffs' not in f:
                raise KeyError('Missing /processed/W_pca_coeffs')
            self.length = int(f['/processed/gcc_phat'].shape[0])

        self._h5 = None

    def _ensure_open(self):
        if self._h5 is None:
            self._h5 = h5py.File(str(self.h5_path), 'r')
        return self._h5

    def __len__(self):
        return self.length

    def __getitem__(self, idx: int):
        f = self._ensure_open()

        x_gcc = np.asarray(f['/processed/gcc_phat'][idx], dtype=np.float32)
        x_s_pca = np.asarray(f['/processed/S_pca_coeffs'][idx], dtype=np.float32)
        y_w_pca = np.asarray(f['/processed/W_pca_coeffs'][idx], dtype=np.float32)

        # 按需求统一到指定训练维度
        x_gcc = resize_1d_sequence(x_gcc, target_len=self.gcc_len)
        x_s_pca = fit_feature_dim(x_s_pca, target_dim=self.s_pca_dim)
        y_w_pca = fit_feature_dim(y_w_pca, target_dim=self.w_pca_dim)

        return (
            torch.from_numpy(x_gcc),
            torch.from_numpy(x_s_pca),
            torch.from_numpy(y_w_pca),
        )

    def __del__(self):
        if getattr(self, '_h5', None) is not None:
            try:
                self._h5.close()
            except Exception:
                pass

In [ ]:
# ===============================
# 构建 DataLoader 与加载 V_w
# ===============================
# 优先使用你的 500 房间数据；若不存在则回退到 dataset.h5
candidate_path = Path('../python_scripts/cfxlms_qc_dataset_500.h5')
DATASET_H5 = candidate_path if candidate_path.exists() else Path('dataset.h5')
print(f'DATASET_H5 = {DATASET_H5.resolve()}')

def load_v_w_tensor(h5_path: Path, device: torch.device) -> torch.Tensor:
    """
    按需求从 /processed/V_w 读取。
    兼容旧数据：若没有 /processed/V_w，则尝试从 /processed/global_svd/W_components 复制一份。
    """
    with h5py.File(str(h5_path), 'r+') as f:
        if '/processed/V_w' in f:
            v_w = np.asarray(f['/processed/V_w'], dtype=np.float32)
        elif '/processed/global_svd/W_components' in f:
            v_w = np.asarray(f['/processed/global_svd/W_components'], dtype=np.float32)
            if 'V_w' not in f['/processed']:
                f['/processed'].create_dataset('V_w', data=v_w.astype(np.float32))
        else:
            raise KeyError('Neither /processed/V_w nor /processed/global_svd/W_components exists.')

    # 训练按 [32, 1536] 使用
    v_w = fit_matrix_shape(v_w, target_rows=32, target_cols=1536)
    v_w_t = torch.tensor(v_w, dtype=torch.float32, device=device)
    return v_w_t

dataset = ANCHDF5Dataset(
    h5_path=DATASET_H5,
    gcc_len=256,
    s_pca_dim=16,
    w_pca_dim=32,
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

batch_size = 64
train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

V_w = load_v_w_tensor(DATASET_H5, device=device)
print(f'Dataset size: {len(dataset)}, train: {len(train_set)}, val: {len(val_set)}')
print(f'V_w shape: {tuple(V_w.shape)}')

x_gcc_b, x_s_b, y_b = next(iter(train_loader))
print('Batch x_gcc:', tuple(x_gcc_b.shape))
print('Batch x_s_pca:', tuple(x_s_b.shape))
print('Batch y_w_pca:', tuple(y_b.shape))

In [ ]:
# =========================================
# 模型定义: PhysicalConditionedANCNet
# =========================================
class PhysicalConditionedANCNet(nn.Module):
    def __init__(self):
        super().__init__()

        # 分支A: 空间特征编码器 (1D-CNN)
        self.spatial_encoder = nn.Sequential(
            nn.Conv1d(in_channels=3, out_channels=16, kernel_size=7, stride=2, padding=3),
            nn.GroupNorm(1, 16),
            nn.GELU(),

            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, stride=2, padding=2),
            nn.GroupNorm(1, 32),
            nn.GELU(),

            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(1, 64),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )

        # 分支B: 物理条件编码器 (MLP)
        self.condition_encoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.LayerNorm(32),
            nn.GELU(),

            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.GELU(),
        )

        # 融合层与预测头
        self.head = nn.Sequential(
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(p=0.2),

            nn.Linear(128, 64),
            nn.GELU(),

            nn.Linear(64, 32),
        )

    def forward(self, x_gcc: torch.Tensor, x_s_pca: torch.Tensor) -> torch.Tensor:
        z_p = self.spatial_encoder(x_gcc)
        z_s = self.condition_encoder(x_s_pca)
        z = torch.cat([z_p, z_s], dim=1)
        c_pred = self.head(z)
        return c_pred

model = PhysicalConditionedANCNet().to(device)
print(model)

with torch.no_grad():
    c_pred_demo = model(x_gcc_b.to(device), x_s_b.to(device))
print('c_pred demo shape:', tuple(c_pred_demo.shape))

In [ ]:
# =========================================
# 自定义 Physics-Aware Loss + 动态绘图
# =========================================
def physics_aware_loss(
    c_pred: torch.Tensor,
    c_true: torch.Tensor,
    v_w: torch.Tensor,
    lambda_phy: float = 0.1,
):
    mse_coef = F.mse_loss(c_pred, c_true)

    # c @ V_w -> 还原高维滤波器波形
    w_pred = c_pred @ v_w
    w_true = c_true @ v_w
    mse_wave = F.mse_loss(w_pred, w_true)

    total = mse_coef + lambda_phy * mse_wave
    return total, mse_coef.detach(), mse_wave.detach()

def plot_live_losses(train_losses, val_losses):
    clear_output(wait=True)
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss', linewidth=2.0)
    plt.plot(val_losses, label='Val Loss', linewidth=2.0)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Physics-Conditioned ANC Net 训练曲线')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
# =========================================
# 训练循环
# =========================================
epochs = 100
lambda_phy = 0.1
lr = 1e-3

model = PhysicalConditionedANCNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=5,
    factor=0.5,
)

best_val_loss = float('inf')
best_model_path = Path('best_model.pth')

train_losses = []
val_losses = []

for epoch in range(1, epochs + 1):
    model.train()
    train_loss_sum = 0.0

    for x_gcc, x_s_pca, y_w_pca in train_loader:
        x_gcc = x_gcc.to(device, non_blocking=True)
        x_s_pca = x_s_pca.to(device, non_blocking=True)
        y_w_pca = y_w_pca.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        c_pred = model(x_gcc, x_s_pca)
        loss, _, _ = physics_aware_loss(c_pred, y_w_pca, V_w, lambda_phy=lambda_phy)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += loss.item() * x_gcc.size(0)

    train_loss_epoch = train_loss_sum / len(train_loader.dataset)

    model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():
        for x_gcc, x_s_pca, y_w_pca in val_loader:
            x_gcc = x_gcc.to(device, non_blocking=True)
            x_s_pca = x_s_pca.to(device, non_blocking=True)
            y_w_pca = y_w_pca.to(device, non_blocking=True)

            c_pred = model(x_gcc, x_s_pca)
            val_loss, _, _ = physics_aware_loss(c_pred, y_w_pca, V_w, lambda_phy=lambda_phy)
            val_loss_sum += val_loss.item() * x_gcc.size(0)

    val_loss_epoch = val_loss_sum / len(val_loader.dataset)

    scheduler.step(val_loss_epoch)

    if val_loss_epoch < best_val_loss:
        best_val_loss = val_loss_epoch
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            },
            best_model_path,
        )

    train_losses.append(train_loss_epoch)
    val_losses.append(val_loss_epoch)

    plot_live_losses(train_losses, val_losses)

    current_lr = optimizer.param_groups[0]['lr']
    print(
        f'Epoch [{epoch:03d}/{epochs}] | '
        f'Train: {train_loss_epoch:.6f} | Val: {val_loss_epoch:.6f} | '
        f'Best Val: {best_val_loss:.6f} | LR: {current_lr:.2e}'
    )

print('训练完成。')
print(f'最优模型已保存到: {best_model_path.resolve()}')

In [ ]:
# =========================================
# 加载最佳模型并做一次简单验证
# =========================================
ckpt = torch.load(best_model_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

val_mse = 0.0
with torch.no_grad():
    for x_gcc, x_s_pca, y_w_pca in val_loader:
        x_gcc = x_gcc.to(device)
        x_s_pca = x_s_pca.to(device)
        y_w_pca = y_w_pca.to(device)

        c_pred = model(x_gcc, x_s_pca)
        val_mse += F.mse_loss(c_pred, y_w_pca, reduction='sum').item()

val_mse /= len(val_loader.dataset)
print(f'Best checkpoint epoch: {ckpt["epoch"]}')
print(f'Checkpoint val loss: {ckpt["val_loss"]:.6f}')
print(f'Validation MSE (coef space): {val_mse:.6f}')

# 演示一条样本的高维滤波器重构
x_gcc, x_s_pca, y_true = val_set[0]
with torch.no_grad():
    y_pred = model(x_gcc.unsqueeze(0).to(device), x_s_pca.unsqueeze(0).to(device)).squeeze(0)

w_true = y_true.to(device) @ V_w
w_pred = y_pred @ V_w
print(f'Reconstructed waveform shape: {tuple(w_pred.shape)}')
print(f'||w_pred - w_true||_2 = {torch.norm(w_pred - w_true).item():.6f}')